# Single-region zero-shot transfer (Sec. 3.2; Fig. 4, A5)

Trains one LSTM and one Transformer on each source region (CH, FR, NOR, ISL) and applies them, without any local data, to every other region.

| Paper figure | Content | Section | Saved as (in `figures/paperTF/`) |
|---|---|---|---|
| Fig. 4 | Δ RMSE (LSTM − Transformer) per source and target, annual and winter, and example scatter plots | Fig. 4 | `fig4_zeroshot_comparison_subpart.png` + `fig4_zeroshot_comparison_subpart_legend.png` (combined outside the code) |
| Fig. A5 | Absolute RMSE of both models per source and target, annual and winter | Fig. A5 | `appendix/A5_zs_comparison_absolute.png` |

**Steps**
1. Load the stake datasets and monthly inputs, and split Central Asia, Alaska and Italy/Austria into sub-regions.
2. For each source region, build a training dataset (all its stakes) and one test dataset per target region.
3. Train (or load) one LSTM and one Transformer per source region.
4. Evaluate every source model on every target and make the figures.
5. Compare the input distributions of a source and a target region (not in the paper).

Datasets and models are cached in `cfg.dataPath/cache/TF_Europe/CROSS_REGION/`. The functions used here live in `scripts/models/cross_region.py` and `scripts/plotting/domain_shift.py`.

## Setting up:

In [ ]:
import os

# Silence KeOps (used by geomloss for Sinkhorn distances) before it is imported
os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

import torch

# Initialise CUDA before the other imports: changing this order can change the
# GPU results slightly (~1e-7)
if torch.cuda.is_available():
    torch.cuda.init()

import sys
import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import massbalancemachine as mbm

sys.path.append(os.path.join(os.getcwd(), '../../'))  # Add root of repo to import MBM
# Shared config (paths, MONTHLY_COLS, STATIC_COLS, REGION_LABELS, COLORS, ...),
# model settings (PARAMS_LSTM, PARAMS_TRANSFORMER, N_EPOCHS, MODEL_DATE) and helpers
from papers.Transformer_TF.scripts.config_TF_Europe import *
from papers.Transformer_TF.scripts.config_models import *
from papers.Transformer_TF.scripts.dataset import *
from papers.Transformer_TF.scripts.plotting import *
from papers.Transformer_TF.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

# Call config file (data paths, seed, etc.)
cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)  # reproducibility
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

# Use the GPU if one is visible
device = torch.device("cuda" if torch.cuda.device_count() > 0 else "cpu")
print(f"Device: {device}")

# Cache folder for the datasets and models of this notebook
BASE_DIR = Path(cfg.dataPath) / path_cache / "CROSS_REGION"
os.makedirs(BASE_DIR, exist_ok=True)
print(f"Base directory for data: {BASE_DIR}")

## Data:
### Stakes and monthly inputs:
With `run_flag_by_code = False`, the monthly data are loaded from `WGMS/CrossRegional/monthly_cache/` instead of being recomputed.

In [ ]:
# Stake datasets of all regions configured in RGI_REGIONS (one DataFrame per RGI region)
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

# Add the RGI O2 sub-region of each glacier in Central Asia and Alaska
dfs, glacier_dict_ca = add_o2region_to_dfs(dfs, '13', cfg)
dfs, glacier_dict_alaska = add_o2region_to_dfs(dfs,
                                               '01',
                                               cfg,
                                               deduplicate_glaciers=True)

SOURCE_REGIONS = ['CH', 'NOR', 'ISL', "FR"]
TARGET_REGIONS = ['IT_AT', 'SJM', 'CENTRALASIA', 'ALA']

run_flag_by_code = {code: False for code in SOURCE_REGIONS + TARGET_REGIONS}  # True = recompute

monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=VOIS_CLIMATE,
    vois_topographical=STATIC_COLS,
    region_codes=SOURCE_REGIONS + TARGET_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

### Sub-regions and source–target pairs:
Italy/Austria is split by country and Central Asia and Alaska by RGI O2 sub-region. Targets are Italy, Austria, Svalbard, Central Asia SR 3 and 4 and Alaska, plus the other source regions.

In [ ]:
# O2 sub-region labels in the monthly data of Central Asia and Alaska
monthly_cache = add_o2region_to_monthly_cache(monthly_cache, 'CENTRALASIA',
                                              glacier_dict_ca)
monthly_cache = add_o2region_to_monthly_cache(monthly_cache, 'ALA',
                                              glacier_dict_alaska)

# Italy / Austria: split by country
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'IT_AT',
    subregions={
        'IT': IT_GLACIERS,
        'AT': AT_GLACIERS
    },
    drop_parent=True,
)

# Central Asia sub-regions
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'CENTRALASIA',
    subregions={
        'CA_12': {
            'o2region_col': 'O2Region',
            'values': ['1', '2']
        },
        'CA_3': {
            'o2region_col': 'O2Region',
            'values': ['3']
        },
        'CA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
    },
)

# Alaska sub-regions
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'ALA',
    subregions={
        'ALA_2': {
            'o2region_col': 'O2Region',
            'values': ['2']
        },
        'ALA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
        'ALA_6': {
            'o2region_col': 'O2Region',
            'values': ['6']
        },
    },
)

# Target regions (with sub-regions)
TARGET_REGIONS_SUB = ['IT', 'AT', 'SJM', 'CA_3', 'CA_4', 'ALA']

# One pair per source region: (source, all other regions)
XREG_PAIRS = [(src,
               [r for r in SOURCE_REGIONS + TARGET_REGIONS_SUB if r != src])
              for src in SOURCE_REGIONS]

# Source (train) and target (test) data per pair, from the monthly cache
res_xreg_by_source, split_info_by_source = prepare_xreg_pairs_from_cache(
    cfg=cfg,
    monthly_cache=monthly_cache,
    xreg_pairs=XREG_PAIRS,
)

## Models:
### Model datasets:
One training dataset per source region and one test dataset per target, keyed `XREG_<source>_TO_<target>` (cached in `datasets/<source>/`).

In [ ]:
logs_cache_dir = BASE_DIR / "datasets"

report_nan_in_xreg_inputs(res_xreg_by_source, MONTHLY_COLS, STATIC_COLS)

outputs_xreg_by_source = {}
for src_region in SOURCE_REGIONS:
    target_codes = [
        t for t in SOURCE_REGIONS + TARGET_REGIONS_SUB if t != src_region
    ]
    outputs_xreg_by_source[src_region] = build_or_load_lstm_all_xreg(
        cfg=cfg,
        res_xreg=res_xreg_by_source[src_region],
        MONTHLY_COLS=MONTHLY_COLS,
        STATIC_COLS=STATIC_COLS,
        cache_dir=logs_cache_dir / src_region,
        ch_code=src_region,
        target_source_codes=target_codes,
        region_groups={},  # no group regions anymore
    )
    print(f"{src_region} -> LSTM dataset keys:",
          list(outputs_xreg_by_source[src_region].keys()))
    report_nan_in_xreg_datasets(src_region, outputs_xreg_by_source[src_region])

### Train models:
One LSTM and one Transformer per source region, trained on all its stakes. With `TRAIN_FLAG = False` the saved models in `models/<source>/` are loaded. When retraining, the LSTMs are always retrained (`force_retrain=True`), while existing Transformers are reused.

In [ ]:
TRAIN_FLAG = False  # set to True to retrain from scratch

# LSTM
models_by_source_lstm, infos_by_source_lstm = train_or_load_source_models(
    cfg,
    device,
    SOURCE_REGIONS,
    outputs_xreg_by_source,
    params=PARAMS_LSTM,
    models_dir=BASE_DIR / "models",
    prefix="lstm_xreg",
    model_type="lstm",
    epochs=N_EPOCHS,
    date=MODEL_DATE,
    train_flag=TRAIN_FLAG,
    force_retrain=True,
)

# Transformer
models_by_source_transformer, infos_by_source_transformer = train_or_load_source_models(
    cfg,
    device,
    SOURCE_REGIONS,
    outputs_xreg_by_source,
    params=PARAMS_TRANSFORMER,
    models_dir=BASE_DIR / "models",
    prefix="transformer_xreg",
    model_type="transformer",
    epochs=N_EPOCHS,
    date=MODEL_DATE,
    train_flag=TRAIN_FLAG,
    force_retrain=False,
)

## Comparison of models:
### Metrics:
Metrics of every source model on every target (Central Asia as a whole is excluded; its sub-regions are kept).

In [ ]:
# Display order of the targets (others follow)
DISPLAY_ORDER = ['CH', 'FR', 'IT', 'NOR', 'ISL', 'SJM', 'ALA', 'CENTRALASIA']
EXCLUDE_TARGETS = {'CENTRALASIA'}

df_metrics_by_source_lstm = evaluate_xreg_metrics(
    cfg, device, models_by_source_lstm, outputs_xreg_by_source,
    SOURCE_REGIONS, EXCLUDE_TARGETS, DISPLAY_ORDER)
df_metrics_by_source_transformer = evaluate_xreg_metrics(
    cfg, device, models_by_source_transformer, outputs_xreg_by_source,
    SOURCE_REGIONS, EXCLUDE_TARGETS, DISPLAY_ORDER)

In [ ]:
# Metrics shown in Fig. 4 and A5 (one row of panels each)
metrics = ["RMSE_annual", "RMSE_winter"]

# Order of the sources (columns) and targets (x axis) in the figures
SOURCE_ORDER = ["CH", "FR", "NOR", "ISL", "ALA", "SJM"]
TARGET_ORDER = ["CH", "FR", "IT", "AT", "NOR", "ISL", "ALA", "SJM"]

df_all, df_delta, sources, targets_per_source = build_xreg_metric_tables(
    df_metrics_by_source_lstm,
    df_metrics_by_source_transformer,
    metrics,
    source_order=SOURCE_ORDER,
    target_order=TARGET_ORDER,
)

### Fig. A5: absolute RMSE

In [ ]:
fig = plot_zs_absolute_rmse(
    df_all,
    metrics,
    sources,
    targets_per_source,
    out_path="figures/paperTF/appendix/A5_zs_comparison_absolute.png",
)

### Fig. 4: Δ RMSE and example scatter plots
The figure and its legend are saved separately and combined into Fig. 4 outside the code.

In [ ]:
# Source -> target pairs shown as scatter plots
pairs = [
    ("CH", "ISL"),
    ("FR", "NOR"),
    ("NOR", "ALA"),
    ("ISL", "IT"),
]

fig = plot_zeroshot_combined(
    cfg,
    device,
    df_delta,
    metrics,
    sources,
    targets_per_source,
    pairs,
    models_by_source_lstm=models_by_source_lstm,
    models_by_source_tf=models_by_source_transformer,
    outputs_xreg_by_source=outputs_xreg_by_source,
    out_path="figures/paperTF/fig4_zeroshot_comparison_subpart.png",
    legend_out_path="figures/paperTF/fig4_zeroshot_comparison_subpart_legend.png",
)

## Domain shift (not in the paper):
### Datasets per target:
Monthly data of each source→target pair, used to compare the input distributions of a source and a target region.

In [ ]:
ALL_TARGETS = SOURCE_REGIONS + TARGET_REGIONS_SUB

res_all_xreg_by_source = {}
for src_region in SOURCE_REGIONS:
    target_codes = [t for t in ALL_TARGETS if t != src_region]

    res_all_xreg_by_source[src_region] = build_xreg_res_all(
        res_xreg=res_xreg_by_source[src_region],
        target_source_codes=target_codes,
        source_col="SOURCE_CODE",
        key_prefix=f"XREG_{src_region}_TO",
        ch_code=src_region,
        region_groups={},  # no group regions anymore
    )
    print(f"{src_region} -> keys:", list(res_all_xreg_by_source[src_region].keys()))

### KDE plots:
Distribution of each input variable and of PMB in a source region (its training data) and a target region.

In [ ]:
# All inputs and the target (ELEVATION_DIFFERENCE is a monthly input, not static)
selected_cols = MONTHLY_COLS + [
    c for c in STATIC_COLS if c != "ELEVATION_DIFFERENCE"
] + ["POINT_BALANCE"]

src = 'CH'
target = 'ISL'
glaciers_to_plot = {
    # source training data (the same for every key, CA_4 is just one of them)
    src: {
        "df": res_all_xreg_by_source[src][f'XREG_{src}_TO_CA_4']["df_train"],
        "color": COLORS[src],
    },
    target: {
        "df": res_all_xreg_by_source[src][f'XREG_{src}_TO_{target}']["df_test"],
        "color": COLORS[target],
    },
}

fig = plot_kde_pair_reg(
    glaciers_to_plot=glaciers_to_plot,
    selected_cols=selected_cols,
)